# Lab 21 — Fine-tuning LLMs · RUN ALL (A100 / BIGGPU)

Thí nghiệm cá nhân: **Mai Văn Trường (2A202602983)**
- Base model: **Qwen/Qwen3.5-9B** (COMPUTE_TIER=BIGGPU)
- Dataset: 250 ticket CSKH -> JSON triage (mặc định)
- Chế độ: **Full run** (EVAL_LIMIT="" - nộp bài chính thức)
- Thời gian ước tính trên A100: **~40 phút**

Chạy từ trên xuống. Runtime > Change runtime type > **A100 GPU** trước khi bắt đầu.

| Ô | Làm gì | Thời gian (A100) |
|---|---|---|
| 1 | clone repo cá nhân + install | ~1 phút |
| 2 | smoke: import + unit test | ~30 giây |
| 3 | **core pipeline NB1 -> NB5** | ~40 phút |
| 4 | gatekeeper + in kết quả | ~10 giây |


In [ ]:
# @title 1. Setup — clone repo cá nhân + install (chạy ô này trước)
import os, subprocess, sys

REPO = "https://github.com/MaiVanTruong2310/Day21-Track3-MaiVanTruong-2A202602983.git"
REPO_DIR = "Day21-Track3-MaiVanTruong-2A202602983"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir(f"/content/{REPO_DIR}")
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

# Cài đặt thư viện theo requirements.txt
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import torch
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"], capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — Runtime > Change runtime type > A100 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


In [ ]:
# @title 2. Smoke — imports, seed data, unit tests (no GPU needed)
!python scripts/verify.py --smoke


In [ ]:
# @title 3. Core pipeline — NB1 → NB5 (A100 / BIGGPU)
# EVAL_LIMIT: "" = full run (submittable), 8 = fast smoke pass.
import os
COMPUTE_TIER = "BIGGPU"        # @param ["CPU","LAPTOP","T4","BIGGPU"]
EVAL_LIMIT   = ""              # @param ["", "4", "8", "16", "25"]
STAGES       = "nb1 nb2 nb3 nb4 nb5"   # @param {type:"string"}

os.environ["COMPUTE_TIER"] = COMPUTE_TIER
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device
print(device.banner(), "\n")

!python scripts/colab_run.py {STAGES}


In [ ]:
# @title 4. Gatekeeper + results
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null
